# Exploartoy Data analysis of the WeatherBench2 Data
In this notebook the ensemble forecasts, as well as the HRES forecast and the ground truth data (HRES_t0) are analyzed.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from genpp.data.weatherbench2 import (
    FORECAST_ENS_PATH,
    OBSERVATIONS_FLAT_PATH,
    FORECAST_ENS_FLAT_AGG_PATH,
)
import xarray as xr

from dask.distributed import Client

client = Client()

client.dashboard_link

In [ ]:
def print_dataset_info(ds):
    print("Dataset Information:")
    print("Latitude range:", ds.latitude.min().values, ds.latitude.max().values)
    print("Number of latitude points:", ds.latitude.size)
    print("Longitude range:", ds.longitude.min().values, ds.longitude.max().values)
    print("Number of longitude points:", ds.longitude.size)
    print("Time range:", ds.time.min().values, ds.time.max().values)
    print("Number of time steps:", ds.time.size)

## IFS ENS forecasts
- there is one day with null values (2019-10-17T00:00:00.000000000)
- this is the only date where there are null values

In [ ]:
ens = xr.open_dataset(
    FORECAST_ENS_PATH,
    chunks={"time": "auto", "number": -1, "latitude": -1, "longitude": -1, "level": -1},
)
ens

In [ ]:
print_dataset_info(ens)

In [ ]:
ens.isnull().sum().compute()

In [ ]:
# Lets investigate the null values in the ensemble dataset
# for some features there are 57350 null values, which corresponds to exaxtly one time step
# print all time steps with null values for the feature 't2m'
null_mask = ens.isnull().compute()
null_times = ens.where(null_mask, drop=True).time
print("Time steps with null values:", null_times.values)

## Observations
- Carries the observations at 00:00 06:00 12:00 and 18:00
- Double the amount of observations as the forecast datasets

In [ ]:
obs = xr.open_dataset(
    OBSERVATIONS_FLAT_PATH,
    chunks="auto",
)
obs

In [ ]:
print_dataset_info(obs)

In [ ]:
obs.isnull().sum().compute()

## Flattended Forecast

In [ ]:
fc_flat = xr.open_dataset(
    FORECAST_ENS_FLAT_AGG_PATH,
    chunks="auto",
)

In [ ]:
fc_flat

## Explore the viability of xbatcher as a direct dataloader
- This process of lazily workig with data and only evaluating when calling the dataloader seems to be rather slow (2:40 min for one epoch only to load the data)
- Next steps (Done):
    - Try saving the intermediate results as zarr or netcdf
    - NetCDF: 12s ✅
    - Zarr: 26s



In [ ]:
from tqdm import tqdm
from torch.utils.data import DataLoader
from genpp.data import OUTPUT_DIR
from genpp.data.dataset import _get_MapDataset
import xarray as xr

In [ ]:
flat_ens_aggr = xr.open_dataarray(
    OUTPUT_DIR / "ens_flat_agg.nc",
    chunks={
        "prediction_time": "auto",
        "latitude": -1,
        "longitude": -1,
        "feature": -1,
    },
)


flat_obs = xr.open_dataarray(
    OUTPUT_DIR / "obs_flat.nc",
    chunks={
        "time": "auto",
        "latitude": -1,
        "longitude": -1,
        "feature": -1,
    },
)

In [ ]:
# This works to produce tiny patches of the data
map_ds = _get_MapDataset(
    x_ds=flat_ens_aggr,
    y_ds=flat_obs,
    x_kwargs={
        "input_dims": {
            "feature": 29 * 2,  # mean and std
            "latitude": 31,
            "longitude": 37,
        },
        "batch_dims": {"prediction_time": 8},
    },
    y_kwargs={
        "input_dims": {"feature": 2, "latitude": 31, "longitude": 37},
        "batch_dims": {"time": 8},
    },
)

In [ ]:
train_loader = DataLoader(
    map_ds,
    batch_size=None,  # This is determined by the batch size of the generators
    shuffle=True,
    prefetch_factor=4,
    num_workers=4,
    multiprocessing_context="forkserver",
)

In [ ]:
# This works just fine
for _ in range(2):
    for batch in tqdm(train_loader):
        continue

In [ ]:
x, y = batch
print(f"{x.shape=}, {y.shape=}")

# Some More Tests

In [ ]:
from genpp.preproc.transforms import Pad
from genpp.preproc.preprocessors import StandardScalerPreprocessor, AddMetadataPreprocessor
from genpp.data import MetadataVars

In [ ]:
add_meta_scale = AddMetadataPreprocessor(
    meta_features=[MetadataVars.LATITUDE, MetadataVars.LONGITUDE]
)
scaler = StandardScalerPreprocessor(dim=["prediction_time", "latitude", "longitude"])
add_meta = AddMetadataPreprocessor(
    meta_features=[
        MetadataVars.SIN_PREDICTION_TIME,
        MetadataVars.COS_PREDICTION_TIME,
        MetadataVars.PIXEL_IDX,
    ]
)


flat_ens_aggr = add_meta_scale.preprocess(flat_ens_aggr)
scaler.fit(flat_ens_aggr)
flat_ens_aggr = scaler.preprocess(flat_ens_aggr)
flat_ens_aggr = add_meta.preprocess(flat_ens_aggr)

flat_ens_aggr = flat_ens_aggr.compute()

In [ ]:
flat_ens_aggr.shape

In [ ]:
# This works to produce tiny patches of the data
map_ds = _get_MapDataset(
    x_ds=flat_ens_aggr,
    y_ds=flat_obs,
    x_kwargs={
        "input_dims": {
            "latitude": 31,
            "longitude": 37,
            "feature": 29 * 2,  # mean and std
        },
        "batch_dims": {"prediction_time": 32},
    },
    y_kwargs={
        "input_dims": {"latitude": 31, "longitude": 37, "feature": 2},
        "batch_dims": {"time": 32},
    },
    x_transform=Pad(padding=(0, 1, 5, 6), mode="constant"),
)

In [ ]:
train_loader = DataLoader(
    map_ds,
    batch_size=None,  # This is determined by the batch size of the generators
    shuffle=True,
    prefetch_factor=3,
    num_workers=4,
    multiprocessing_context="forkserver",
)

In [ ]:
for _ in range(1):
    for batch in tqdm(train_loader):
        continue

In [ ]:
print(batch[0].shape)
print(batch[1].shape)

In [ ]:
from pathlib import Path

p = Path("/Users/moritzfeik/Developer/GENPP/src/genpp/data/flat_ens_preproc_agg.nc")
# To append something to the filename (before the extension)
new_p = p.with_name(p.stem + "_appended" + p.suffix)
print(new_p)

# Or to append to the entire name (including extension)
new_p2 = p.with_name(p.name + "_appended")
print(new_p2)